# 9-тарау. Диффузиялық модельдер: шудан суретке

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 9.1. Сия тамшысы және уақытты кері айналдыру

## 9.2. Деректер: «ЖИ» әріптері

In [ ]:
import math
import torch
from torch import nn

SEGMENTS = [((-1.3, -1), (-1.3, 1)), ((-1.3, 0), (-2.1, 1)),     # Ж
            ((-1.3, 0), (-2.1, -1)), ((-1.3, 0), (-0.5, 1)),
            ((-1.3, 0), (-0.5, -1)),
            ((0.6, -1), (0.6, 1)), ((2.0, -1), (2.0, 1)),         # И
            ((0.6, -1), (2.0, 1))]
A = torch.tensor([s[0] for s in SEGMENTS])
B = torch.tensor([s[1] for s in SEGMENTS])
lengths = (B - A).norm(dim=1)

def sample_letters(n):
    k = torch.multinomial(lengths, n, replacement=True)   # кесінді
    t = torch.rand(n, 1)                                  # бойындағы орны
    pts = A[k] + t * (B[k] - A[k]) + 0.04 * torch.randn(n, 2)
    return 0.5 * pts                                      # шамамен [-1; 1]

torch.manual_seed(0)
data = sample_letters(2000)
print(data.shape, data.min(0).values, data.max(0).values)

## 9.3. Тура процесс: шуды біртіндеп қосу

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.05, T)
alphas = 1 - betas
alpha_bar = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, eps):
    """x0 нүктесін t қадамға дейін шуландыру."""
    a = alpha_bar[t].sqrt().unsqueeze(1)
    b = (1 - alpha_bar[t]).sqrt().unsqueeze(1)
    return a * x0 + b * eps

for t in [0, 20, 50, 100, 199]:
    print(f"t={t:3d}: сигнал={alpha_bar[t].sqrt():.3f}, "
          f"шу={(1 - alpha_bar[t]).sqrt():.3f}")
forward = {t: q_sample(data, torch.full((2000,), t), torch.randn(2000, 2))
           for t in [0, 20, 50, 100, 199]}

## 9.4. Шуды болжайтын желі

In [ ]:
def time_features(t, n=16):
    freqs = torch.exp(torch.linspace(0, math.log(200), n))
    a = t.float().unsqueeze(1) / T * freqs
    return torch.cat([a.sin(), a.cos()], dim=1)          # 32 сан

def point_features(x, n=12):
    freqs = torch.exp(torch.linspace(0, math.log(60), n))
    a = x.unsqueeze(-1) * freqs                          # (N, 2, 12)
    return torch.cat([x, a.sin().flatten(1), a.cos().flatten(1)], dim=1)

class Denoiser(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 + 48 + 32, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 2))                        # болжанған шу

    def forward(self, x, t):
        return self.net(torch.cat([point_features(x), time_features(t)], 1))

In [ ]:
torch.manual_seed(0)
model = Denoiser()
steps = 8000
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
for step in range(1, steps + 1):
    x0 = sample_letters(1024)
    t = torch.randint(0, T, (1024,))
    eps = torch.randn_like(x0)
    loss = ((model(q_sample(x0, t, eps), t) - eps) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    sched.step()
    if step % 2000 == 0:
        print(f"{step}: шығын={loss.item():.3f}")

## 9.5. Кері процесс: шудан әріпке

In [ ]:
@torch.no_grad()
def generate(n, keep=(199, 150, 100, 50, 0)):
    x = torch.randn(n, 2)                         # таза шу
    history = {}
    for t in reversed(range(T)):
        eps_hat = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps_hat)
        x = x / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
        if t in keep:
            history[t] = x.clone()
    return x, history

torch.manual_seed(1)
samples, history = generate(2000)
print(samples.shape)

In [ ]:
def distance_to_letters(p):
    a, b = 0.5 * A, 0.5 * B
    ab = b - a
    ap = p.unsqueeze(1) - a.unsqueeze(0)
    t = ((ap * ab).sum(-1) / (ab * ab).sum(-1)).clamp(0, 1)
    proj = a.unsqueeze(0) + t.unsqueeze(-1) * ab.unsqueeze(0)
    return (p.unsqueeze(1) - proj).norm(dim=-1).min(dim=1).values

for name, pts in [("таза шу", torch.randn(2000, 2)),
                  ("модель", samples), ("шын деректер", data)]:
    close = (distance_to_letters(pts) < 0.05).float().mean()
    print(f"{name:13s}: {close:.0%}")

## 9.6. Нүктелерден суреттерге